In [2]:
import sys
import pandas as pd
import json
import os
from pathlib import Path

# adicionar a pasta src no path
sys.path.insert(0, os.path.abspath('../src'))

from recommendation_engine import RecommendationEngine

print("Imports OK")
print(f"Python: {sys.version}")


Imports OK
Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


In [3]:
print("=" * 80)
print("FASE 6: MODELO DE SIMILARIDADE - TREINAMENTO")
print("=" * 80)

# Paths
products_path = '../data/refined/products_features.parquet'
model_path = '../models/recommendation_engine.pkl'
metadata_path = '../models/model_metadata.json'

# Criar diretórios
Path('../models').mkdir(parents=True, exist_ok=True)
Path('../data/refined').mkdir(parents=True, exist_ok=True)

# Carregar dados
print("\n1. CARREGANDO DADOS")
products = pd.read_parquet(products_path)
print(f"{len(products):,} produtos carregados")
print(f"Colunas: {list(products.columns)}")

# Verificar coluna full_description
if 'full_description' not in products.columns:
    print("Coluna 'full_description' não encontrada!")
    print(f"Colunas disponíveis: {products.columns.tolist()}")
else:
    print(f"Coluna 'full_description' encontrada")
    print(f"Exemplo: {products['full_description'].iloc[0][:100]}...")


FASE 6: MODELO DE SIMILARIDADE - TREINAMENTO

1. CARREGANDO DADOS
10,000 produtos carregados
Colunas: ['product_id', 'product_name', 'product_category', 'product_subcategory', 'manufacturer', 'model', 'bearing_type', 'material', 'load_capacity', 'max_speed', 'temperature_limit', 'problem_type', 'unit_cost', 'list_price', 'technical_description', 'technical_features', 'llm_product_description', 'supported_problems', 'problem_Contaminação', 'problem_Desgaste', 'problem_Superaquecimento', 'problem_Vibração', 'full_description']
Coluna 'full_description' encontrada
Exemplo: Rolamento Rolamento Industrial 1 - Autocompensador em Aço. Problema: ['Vibração']. Descrição: Rolame...


In [4]:
print("\n2. TREINANDO ENGINE")

# Instanciar
engine = RecommendationEngine(random_state=42)

# Treinar
engine.fit(products, text_column='full_description')

# Info
info = engine.get_info()
print(f"Engine treinado!")
print(f"Status: {info['status']}")
print(f"Produtos: {info['num_products']:,}")
print(f"Vocab: {info['vocab_size']:,} termos")
print(f"Params: {info['tfidf_params']}")



2. TREINANDO ENGINE
Engine treinado!
Status: trained
Produtos: 10,000
Vocab: 1,000 termos
Params: {'max_features': 1000, 'stop_words': 'portuguese', 'min_df': 2, 'max_df': 0.8}


In [5]:
print("\n3. TESTANDO RECOMENDAÇÕES")

test_queries = [
    "Máquina vibrando muito, preciso de um rolamento que resolva vibração",
    "Superaquecimento no eixo, qual rolamento suporta alta temperatura?",
    "Desgaste rápido, preciso de durabilidade e longa vida útil",
    "Ambiente úmido e contamição, preciso de vedação",
    "Alta velocidade, baixa vibração, rolamento de precisão"
]

results_summary = {}

for i, query in enumerate(test_queries, 1):
    print(f"\n[{i}] Query: '{query[:60]}...'")
    
    recommendations = engine.recommend(query, top_k=5, min_score=0.1)
    results_summary[query] = recommendations
    
    for j, rec in enumerate(recommendations, 1):
        score_pct = rec['score'] * 100
        print(f"       {j}. {rec['product_name'][:40]:<40} ({score_pct:.1f}%)")

print(f"\nTestadas {len(test_queries)} queries com sucesso")



3. TESTANDO RECOMENDAÇÕES

[1] Query: 'Máquina vibrando muito, preciso de um rolamento que resolva ...'
       1. Rolamento Industrial 9412                (32.8%)
       2. Rolamento Industrial 5637                (32.7%)
       3. Rolamento Industrial 8863                (32.1%)
       4. Rolamento Industrial 7586                (31.6%)
       5. Rolamento Industrial 9454                (31.6%)

[2] Query: 'Superaquecimento no eixo, qual rolamento suporta alta temper...'
       1. Rolamento Industrial 2608                (33.0%)
       2. Rolamento Industrial 3617                (32.6%)
       3. Rolamento Industrial 1972                (32.4%)
       4. Rolamento Industrial 1457                (32.2%)
       5. Rolamento Industrial 1801                (32.1%)

[3] Query: 'Desgaste rápido, preciso de durabilidade e longa vida útil...'
       1. Rolamento Industrial 9788                (32.3%)
       2. Rolamento Industrial 9131                (32.3%)
       3. Rolamento Industrial 27

In [6]:
print("\n4. SALVANDO MODELO")

# Salvar modelo
engine.save_model(model_path)
print(f"Modelo salvo: {model_path}")

# Criar metadata
metadata = {
    'model_type': 'TfidfVectorizer + CosineSimilarity',
    'training_date': pd.Timestamp.now().isoformat(),
    'num_products': len(products),
    'vocab_size': info['vocab_size'],
    'tfidf_params': info['tfidf_params'],
    'test_queries': len(test_queries),
    'status': 'production'
}

# Salvar metadata
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata salvo: {metadata_path}")
print(f"Conteúdo:")
for k, v in metadata.items():
    print(f"       {k}: {v}")



4. SALVANDO MODELO
Modelo salvo: ../models/recommendation_engine.pkl
Metadata salvo: ../models/model_metadata.json
Conteúdo:
       model_type: TfidfVectorizer + CosineSimilarity
       training_date: 2026-01-10T20:05:38.702681
       num_products: 10000
       vocab_size: 1000
       tfidf_params: {'max_features': 1000, 'stop_words': 'portuguese', 'min_df': 2, 'max_df': 0.8}
       test_queries: 5
       status: production


In [7]:
print("\n5. VALIDANDO MODELO CARREGADO")

# Carregar modelo
loaded_engine = RecommendationEngine.load_model(model_path)
print(f"Modelo carregado com sucesso")

# Testar com nova query
test_query = "Vibração e superaquecimento"
loaded_results = loaded_engine.recommend(test_query, top_k=3)

print(f"\n   Teste de integridade (query: '{test_query}'):")
for i, rec in enumerate(loaded_results, 1):
    print(f"       {i}. {rec['product_name'][:40]} ({rec['score']:.3f})")

print(f"\nValidação OK - Modelo está funcional")



5. VALIDANDO MODELO CARREGADO
Modelo carregado com sucesso

   Teste de integridade (query: 'Vibração e superaquecimento'):
       1. Rolamento Industrial 2608 (0.235)
       2. Rolamento Industrial 3617 (0.232)
       3. Rolamento Industrial 1972 (0.231)

Validação OK - Modelo está funcional


In [9]:
print("\n" + "=" * 80)
print("RESUMO - FASE 6 CONCLUÍDA")
print("=" * 80)

print(f"""
Modelo Treinado:
   • Arquivo: {model_path}
   • Tamanho: {Path(model_path).stat().st_size / 1024:.1f} KB
   • Produtos: {metadata['num_products']:,}
   • Vocab: {metadata['vocab_size']:,}

Metadata:
   • Arquivo: {metadata_path}
   • Queries testadas: {metadata['test_queries']}
   • Status: {metadata['status']}

""")

print("Arquivos criados:")
print(f"1. {model_path}")
print(f"2. {metadata_path}")




RESUMO - FASE 6 CONCLUÍDA

Modelo Treinado:
   • Arquivo: ../models/recommendation_engine.pkl
   • Tamanho: 16281.1 KB
   • Produtos: 10,000
   • Vocab: 1,000

Metadata:
   • Arquivo: ../models/model_metadata.json
   • Queries testadas: 5
   • Status: production


Arquivos criados:
1. ../models/recommendation_engine.pkl
2. ../models/model_metadata.json
